In [ ]:
import pandas as pd
import re
import jieba
from datetime import datetime

# 读取Excel文件
df = pd.read_excel('data/20250519zwj数据爬取/微博评论（utf-8）.xlsx')

# 清洗数据：去除无关行、处理缺失值
df = df.dropna(subset=['评论内容'])
df = df.iloc[1:]  # 假设第一行是标题行，直接跳过

# 定义分词函数（可自定义停用词）
def tokenize(text):
    stopwords = set(['的', '了', '啊', '呀', '呢', '！', '？', '，', '。'])  # 自定义停用词
    words = jieba.lcut(text)
    return [word for word in words if word not in stopwords]

# 清洗评论内容（去除标点符号）
df['cleaned_评论内容'] = df['评论内容'].apply(lambda x: re.sub(r'[^\w\s]', '', x))

# 提取用户ID、时间、点赞数，并分词
df['user_id'] = df['用户名']
df['post_time'] = df['时间'].apply(lambda x: datetime.strptime(x, '%Y/%m/%d %H:%M:%S') if isinstance(x, str) else None)  # 假设时间格式为'YYYY/MM/DD HH:MM:SS'
df['likes'] = df['点赞数'].apply(lambda x: int(x) if isinstance(x, str) else 0)  # 点赞数转为整数
df['comments_tokenized'] = df['cleaned_评论内容'].apply(tokenize)

# 查看结果
print(df.head())


In [4]:
import chardet

def detect_file_encoding(file_path):
    with open(file_path, 'rb') as file:
        raw_data = file.read()
        result = chardet.detect(raw_data)
        return result

file_path = input("请输入文件路径：")
encoding_info = detect_file_encoding(file_path)
print(f"文件编码：{encoding_info['encoding']}, 置信度：{encoding_info['confidence']}")

请输入文件路径： D:\\anotebook\\期末大作业\\大作业\\微博评论.csv


文件编码：GB2312, 置信度：0.99


In [16]:
import pandas as pd
import re

# 读取CSV文件（假设文件名为'weibo_comments.csv'）
df = pd.read_csv('微博评论.csv',encoding='gbk', on_bad_lines='skip', dtype={'时间': str, '评论内容': str})

# 清洗数据：去除无关行、处理缺失值
df = df.dropna(subset=['评论内容'])
df = df[df['评论内容'] != '评论内容']  # 去除标题行

import jieba
from datetime import datetime

# 定义分词函数（可自定义停用词）
def tokenize(text):
    stopwords = set(['的', '了', '啊', '呀', '呢', '！', '？', '，', '。'])  # 自定义停用词
    words = jieba.lcut(text)
    return [word for word in words if word not in stopwords]

# 提取用户ID、时间、点赞数，并分词
df['时间'] = df['时间'].astype(str)
df['post_time'] = df['时间'].apply(lambda x: datetime.strptime(x, '%Y/%m/%d %H:%M') if x != 'nan' else None)
df['user_id'] = df['用户名']
df['likes'] = df['点赞数'].apply(lambda x: int(x) if isinstance(x, str) else 0)  # 点赞数转为整数
df['comments_tokenized'] = df['cleaned_评论内容'].apply(tokenize)  # 假设'cleaned_评论内容'为清洗后的文本


AttributeError: 'float' object has no attribute 'decode'

In [22]:
from afinn import Afinn

# 初始化 AFINN 模型（默认使用英文词典）
afinn = Afinn(language='en')  # 若有中文词典，可设置为 'zh'

# 定义情感分析函数（支持中文需自定义词典）
def afinn_sentiment(text):
    score = afinn.score(text)
    if score > 0:
        return 'positive'
    elif score < 0:
        return 'negative'
    else:
        return 'neutral'

# 对评论内容进行情感分析
df['cleaned_评论内容'] = df['cleaned_评论内容'].astype(str)
df['sentiment_afinn'] = df['cleaned_评论内容'].apply(afinn_sentiment)

In [31]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
import pandas as pd

# 读取训练数据
train_data = pd.read_csv('sentiment_train.csv',encoding='gbk')
X_train = train_data['text']
y_train = train_data['label']

# 创建并训练模型
pipeline = Pipeline([
    ('tfidf', TfidfVectorizer()),
    ('clf', LogisticRegression())
])
pipeline.fit(X_train, y_train)

# 定义情感预测函数
def ml_sentiment(text):
    return 'positive' if pipeline.predict([text])[0] == 1 else 'negative'

# 对评论内容进行情感分析
df['sentiment_ml'] = df['cleaned_评论内容'].apply(ml_sentiment)

In [32]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

# 假设已有标注好的训练数据（需提前准备）
# X_train, y_train = ...  # 训练数据特征和标签
# pipeline = Pipeline([
#     ('tfidf', TfidfVectorizer()),
#     ('clf', LogisticRegression())
# ])
# pipeline.fit(X_train, y_train)

# 预测情感（示例代码，需替换为实际训练好的模型）
def ml_sentiment(text):
    # 实际应用中需将文本转换为特征向量后输入模型
    return 'positive' if pipeline.predict([text])[0] == 1 else 'negative'

df['sentiment_ml'] = df['cleaned_评论内容'].apply(ml_sentiment)

In [ ]:
from transformers import BertTokenizer, BertForSequenceClassification
import torch

# 加载预训练模型（示例使用bert-base-chinese）
tokenizer = BertTokenizer.from_pretrained('bert-base-chinese')
model = BertForSequenceClassification.from_pretrained('bert-base-chinese', num_labels=3)  # 三分类：积极、中性、消极

def bert_sentiment(text):
    inputs = tokenizer(text, return_tensors='pt', padding=True, truncation=True, max_length=512)
    with torch.no_grad():
        outputs = model(**inputs)
        scores = torch.softmax(outputs.logits, dim=1).squeeze().tolist()
    return ['neutral', 'negative', 'positive'][scores.index(max(scores))]  # 假设索引0-2对应中性、消极、积极

df['sentiment_bert'] = df['cleaned_评论内容'].apply(bert_sentiment)

In [ ]:
import pandas as pd
# 这里假设文件名为'weibo_comments.csv'，根据实际情况修改
df = pd.read_csv('情感分析.csv', encoding='utf-8', on_bad_lines='skip') 
# 输出前5条结果
print(df[['user_id', 'post_time', 'likes', 'comments_tokenized', 'sentiment_afinn', 'sentiment_ml', 'sentiment_bert']].head())

# 保存结果到CSV
df.to_csv('情感分析.csv', index=False, encoding='utf-8')